# Demonstrativo: Algoritmo de Huffman na Transmissão de Telemetria SCADA (Aqua Index)

Este notebook demonstra a implementação e simulação do **Algoritmo de Huffman** para compressão sem perdas (*lossless*) de dados de telemetria industrial no sistema **SCADA Aqua Index** da **Máquina Pneumática de Envasamento de Água em Copos**.

---

## 1. Contexto Operacional e Mapeamento de Tags ISA-5.1

Na arquitetura **Cliente-Servidor** da envasadora (Automation Studio / CLP enviando status para o portal SCADA), o vetor de alarmes críticos das estações e do painel geral é composto por **5 sinais booleanos** (`0` = condição nominal, `1` = alarme/intertravamento ativo):

1. `ZS-102` (Setor 100 - E1): Sensor capacitivo de dispensa (`1` = Ausência de Copo / Magazine Vazio — `R-04`)
2. `ZSC-203` (Setor 200 - E2): Sensor magnético do bico dosador (`1` = Bico Fechado / Falha de Dosagem — `R-05`)
3. `PIT-301` (Setor 300 - E3): Pressostato de vácuo da ventosa (`1` = Falha de Vácuo / Tampa não Capturada — `R-03`)
4. `TIT-401` (Setor 400 - E4): Sensor térmico da termosselagem (`1` = Selagem Fria, $T < 180^\circ\text{C}$ — `R-02`)
5. `ESD-001` (Setor 000 - Geral): Botão de Parada de Emergência (`1` = Emergência Acionada — `e1`)

Sem compressão, cada leitura de telemetria exige uma palavra binária de tamanho fixo de **5 bits**.

In [ ]:
import heapq
import math
import random
import matplotlib.pyplot as plt
import pandas as pd

# Configuração visual dos gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
print("Bibliotecas carregadas com sucesso.")

## 2. Modelo Probabilístico das Mensagens da Envasadora

Em regime produtivo contínuo na mesa giratória indexada, o estado nominal (`00000`) domina amplamente as amostras coletadas, enquanto faltas de insumo (`ZS-102`), falhas de vácuo (`PIT-301`), desvios térmicos (`TIT-401`), falhas no bico (`ZSC-203`) e acionamentos de emergência (`ESD-001`) ocorrem com frequências decrescentes.

In [ ]:
# Probabilidades de cada vetor lógico [ZS-102, ZSC-203, PIT-301, TIT-401, ESD-001]
state_probabilities = {
    '00000': 0.880,  # Operação Nominal Estável
    '10000': 0.055,  # Falta de Copo / Magazine Vazio (ZS-102)
    '00100': 0.030,  # Falha de Vácuo / Tampa (PIT-301)
    '00010': 0.020,  # Selagem Fria < 180°C (TIT-401)
    '01000': 0.012,  # Bico Dosador não Confirmado (ZSC-203)
    '00001': 0.003   # Parada de Emergência Acionada (ESD-001)
}

state_descriptions = {
    '00000': 'Operação Nominal Estável',
    '10000': 'E1: Falta de Copo (ZS-102)',
    '00100': 'E3: Falha de Vácuo (PIT-301)',
    '00010': 'E4: Selagem Fria (TIT-401)',
    '01000': 'E2: Falha Bico Dosador (ZSC-203)',
    '00001': 'Geral: Emergência (ESD-001)'
}

df_probs = pd.DataFrame([
    {
        'Vetor de Estado': sym,
        'Condição Operacional': state_descriptions[sym],
        'Probabilidade': p,
        'Porcentagem (%)': round(p * 100, 1)
    }
    for sym, p in state_probabilities.items()
])

print('--- Tabela de Probabilidades de Telemetria — Aqua Index ---')
display(df_probs)

## 3. Construção da Árvore de Huffman e Geração de Códigos

Implementamos a estrutura de árvore binária utilizando uma fila de prioridade (*min-heap*) para gerar códigos livres de prefixo com tamanho variável ótimo.

In [ ]:
class HuffmanNode:
    def __init__(self, symbol=None, freq=0.0, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_tree(probabilities):
    heap = [HuffmanNode(symbol=sym, freq=p) for sym, p in probabilities.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = HuffmanNode(freq=left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)

    return heap[0]


def generate_codes(root, current_code="", code_dict=None):
    if code_dict is None:
        code_dict = {}
    if root is not None:
        if root.symbol is not None:
            code_dict[root.symbol] = current_code
        generate_codes(root.left, current_code + "0", code_dict)
        generate_codes(root.right, current_code + "1", code_dict)
    return code_dict


# Construir árvore e gerar dicionário de códigos prefixados
tree_root = build_huffman_tree(state_probabilities)
huffman_codes = generate_codes(tree_root)

print("--- Códigos de Huffman Gerados (Aqua Index) ---")
for sym in state_probabilities:
    code = huffman_codes[sym]
    print(
        f"Estado: {sym} | Probabilidade: {state_probabilities[sym]:6.3f} | "
        f"Código Huffman: {code:>5} ({len(code)} bit{'s' if len(code) > 1 else ''}) | {state_descriptions[sym]}"
    )

## 4. Simulação de Transmissão SCADA em Tempo Real

Simulamos o envio contínuo de **10.000 amostras** de telemetria geradas pelo processo da envasadora (Automation Studio / CLP) para o cliente supervisório **Aqua Index**.

In [ ]:
# Série temporal simulada de 10.000 ciclos de varredura do SCADA
NUM_SAMPLES = 10000
symbols = list(state_probabilities.keys())
weights = list(state_probabilities.values())

random.seed(42)
telemetry_stream = random.choices(symbols, weights=weights, k=NUM_SAMPLES)

# 1. Transmissão em formato fixo (5 bits por amostra)
raw_bitstream = "".join(telemetry_stream)
raw_size_bits = len(raw_bitstream)

# 2. Transmissão comprimida com Huffman
compressed_bitstream = "".join(huffman_codes[sym] for sym in telemetry_stream)
compressed_size_bits = len(compressed_bitstream)

reduction_pct = 100 * (1 - compressed_size_bits / raw_size_bits)

print(f"Amostras Processadas:           {NUM_SAMPLES:,}")
print(f"Tamanho Bruto (Tamanho Fixo):   {raw_size_bits:,} bits ({raw_size_bits / 8 / 1024:.2f} KB)")
print(f"Tamanho Comprimido (Huffman):   {compressed_size_bits:,} bits ({compressed_size_bits / 8 / 1024:.2f} KB)")
print(f"Redução Experimental de Volume: {reduction_pct:.2f}%")

## 5. Algoritmo de Decodificação e Validação Sem Perdas (*Lossless*)

No receptor do portal SCADA **Aqua Index**, o fluxo binário comprimido é navegado bit a bit pela Árvore de Huffman para recuperar com 100% de fidelidade a sequência original de estados das cinco estações.

In [ ]:
def decode_huffman(bitstream, root):
    decoded_symbols = []
    current = root
    for bit in bitstream:
        current = current.left if bit == '0' else current.right
        if current.symbol is not None:
            decoded_symbols.append(current.symbol)
            current = root
    return decoded_symbols


# Executar decodificação no cliente SCADA
decoded_telemetry = decode_huffman(compressed_bitstream, tree_root)

# Validação formal de integridade (Lossless Verification)
is_equal = (telemetry_stream == decoded_telemetry)
assert is_equal, "Erro na decodificação Huffman: divergência detectada!"
print("Validação de Integridade (Lossless Verification): ✅ SUCESSO! Recuperação 100% fiel das 10.000 amostras.")

## 6. Análise Teórica e Gráficos de Desempenho

Comparamos o comprimento médio obtido pelo código de Huffman com o limite inferior teórico da **Entropia de Shannon** $H(S)$ e com a codificação de comprimento fixo.

In [ ]:
# Entropia de Shannon H(S)
entropy = -sum(p * math.log2(p) for p in state_probabilities.values())

# Comprimento Médio Teórico L
avg_length = sum(state_probabilities[sym] * len(huffman_codes[sym]) for sym in state_probabilities)

print(f"Entropia de Shannon H(S):          {entropy:.4f} bits/símbolo")
print(f"Comprimento Médio da Palavra L:    {avg_length:.4f} bits/símbolo")
print(f"Comprimento do Formato Fixo:       5.0000 bits/símbolo")
print(f"Eficiência Teórica de Compressão:  {(1 - avg_length / 5) * 100:.2f}%")

# Gráficos comparativos de desempenho
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Volume total transmitido
categories = ['Sem Compressão\n(5 bits fixos)', 'Com Compressão\n(Huffman Aqua Index)']
values_kb = [raw_size_bits / 8 / 1024, compressed_size_bits / 8 / 1024]
colors = ['#e74c3c', '#2ecc71']

bars = ax1.bar(categories, values_kb, color=colors, width=0.5)
ax1.set_ylabel('Volume de Dados (KB)')
ax1.set_title('Volume Total Transmitido — Aqua Index (10.000 Amostras)')
ax1.set_ylim(0, max(values_kb) * 1.2)
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, yval + 0.12, f'{yval:.2f} KB', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: Comprimento do código vs. Frequência
states = list(state_probabilities.keys())
probs = [state_probabilities[s] * 100 for s in states]
code_lens = [len(huffman_codes[s]) for s in states]

l1 = ax2.bar(states, code_lens, color='#3498db', alpha=0.8, label='Tamanho do Código (bits)')
ax2_twin = ax2.twinx()
l2 = ax2_twin.plot(states, probs, color='#f39c12', marker='o', linewidth=2.2, label='Probabilidade (%)')

ax2.set_xlabel('Vetor de Estado [ZS-102, ZSC-203, PIT-301, TIT-401, ESD-001]')
ax2.set_ylabel('Comprimento do Código (bits)')
ax2_twin.set_ylabel('Frequência de Ocorrência (%)')
ax2.set_title('Relação Frequência vs. Tamanho do Código Huffman')

plt.tight_layout()
plt.show()